In [52]:
import pandas as pd
import numpy as np
from thefuzz import fuzz, process

In [53]:
url = "https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/main/train.csv"

df = pd.read_csv(url)
df.head()


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


# Data Exploration 

In [54]:
print(df.shape)

(75973, 14)


In [55]:
#df.carID.count() # No duplicates for CarID

In [56]:
df.describe()

,carID,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
count,75973.000000,74482.000000,75973.000000,74510.000000,68069.000000,68047.000000,74457.000000,74449.000000,74423.000000,74425.0
mean,37986.000000,2017.096611,16881.889553,23004.184088,120.329078,55.152666,1.660136,64.590667,1.994580,0.0
std,21931.660338,2.208704,9736.926322,22129.788366,65.521176,16.497837,0.573462,21.021065,1.472981,0.0
min,0.000000,1970.000000,450.000000,-58540.574478,-91.121630,-43.421768,-0.103493,1.638913,-2.345650,0.0
25%,18993.000000,2016.000000,10200.000000,7423.250000,125.000000,46.300000,1.200000,47.000000,1.000000,0.0
50%,37986.000000,2017.000000,14699.000000,17300.000000,145.000000,54.300000,1.600000,65.000000,2.000000,0.0
75%,56979.000000,2019.000000,20950.000000,32427.500000,145.000000,62.800000,2.000000,82.000000,3.000000,0.0
max,75972.000000,2024.121759,159999.000000,323000.000000,580.000000,470.800000,6.600000,125.594308,6.258371,0.0


In [57]:
df.dtypes

carID               int64
Brand              object
model              object
year              float64
price               int64
transmission       object
mileage           float64
fuelType           object
tax               float64
mpg               float64
engineSize        float64
paintQuality%     float64
previousOwners    float64
hasDamage         float64
dtype: object

In [58]:
df.isna().sum()

carID                0
Brand             1521
model             1517
year              1491
price                0
transmission      1522
mileage           1463
fuelType          1511
tax               7904
mpg               7926
engineSize        1516
paintQuality%     1524
previousOwners    1550
hasDamage         1548
dtype: int64

# Data Cleaning

## Cleaning text columns
### Resovling Spelling Issues in the Text columns
When exploring the data we see that there a multiple errors with the spelling of the Brand, model, transmission, fuelType columns. In the next section we will try to resolve that and create a coherent naming.

In [59]:
#df["Brand"] = df["Brand"].str.lower().str.strip() # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column

df["model"] = df["model"].str.lower().str.strip()
df["transmission"] = df["transmission"].str.lower().str.strip()
df["fuelType"] = df["fuelType"].str.lower().str.strip()

df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN") # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)

In [60]:
# Optional display block, commented for compactnes
# Will show all the unqiue values for the text columns
"""print(df["Brand"].unique())
print("\n------------------------------------ \n")
print(df["model"].unique())
print("\n------------------------------------ \n")
print(df["transmission"].unique())
print("\n------------------------------------ \n")
print(df["fuelType"].unique())"""

'print(df["Brand"].unique())\nprint("\n------------------------------------ \n")\nprint(df["model"].unique())\nprint("\n------------------------------------ \n")\nprint(df["transmission"].unique())\nprint("\n------------------------------------ \n")\nprint(df["fuelType"].unique())'

#### Brands

In [61]:
brand_mapping = {
    "vw": "vw",
    "v": "vw",
    "w": "vw",
    
    "toyota": "toyota",
    "toyot": "toyota",
    "oyota": "toyota",
    
    "audi": "audi",
    "aud": "audi",
    "udi": "audi",
    "ud": "audi",
    
    "ford": "ford",
    "for": "ford",
    "ord": "ford",
    "or": "ford",
    
    "bmw": "bmw",
    "bm": "bmw",
    "mw": "bmw",
    
    "skoda": "skoda",
    "skod": "skoda",
    "koda": "skoda",
    "kod": "skoda",
    
    "opel": "opel",
    "ope": "opel",
    "pel": "opel",
    "pe": "opel",
    
    "mercedes": "mercedes",
    "mercede": "mercedes",
    "ercedes": "mercedes",
    "ercede": "mercedes",
    
    "hyundai": "hyundai",
    "hyunda": "hyundai",
    "yundai": "hyundai",
    "yunda": "hyundai"
}

df["Brand"] = df["Brand"].str.lower().map(brand_mapping)

#### Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)

In [62]:
# Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz

# DISCLAMER USE OF AI: We used AI to compile the list of the correctly written car models.
# Linkt to the chat: https://claude.ai/share/501de708-da33-4c2c-8c05-4e21f7facf7a
models = ['golf', 'yaris', 'q2', 'fiesta', '2 series', '3 series', 'a3', 'octavia', 'passat', 'focus', 'insignia', 'a class', 'q3', 'fabia', 'ka+', 'glc class', 'i30', 'c class', 'polo', 'e class', 'q5', 'up', 'c-hr', 'mokka x', 'corsa', 'astra', 'tt', '5 series', 'aygo', '4 series', 'slk', 'viva', 't-roc', 'ecosport', 'tucson', 'x-class', 'cl class', 'ix20', 'i20', 'rapid', 'a1', 'auris', 'sharan', 'adam', 'x3', 'a8', 'gls class', 'b-max', 'a4', 'kona', 'i10', 'mokka', 's-max', 'x2', 'crossland x', 'tiguan', 'a5', 'gle class', 'zafira', 'ioniq', 'a6', 'mondeo', 'yeti outdoor', 'x1', 'scala', 's class', '1 series', 'kamiq', 'kuga', 'tourneo connect', 'q7', 'gla class', 'arteon', 'sl class', 'santa fe', 'grandland x', 'i800', 'rav4', 'touran', 'citigo', 'roomster', 'prius', 'corolla', 'b class', 'kodiaq', 'v class', 'caddy maxi life', 'superb', 'getz', 'combo life', 'beetle', 'galaxy', 'm3', 'gtc', 'x4', 'ka', 'ix35', 'grand tourneo connect', 'm4', 'tourneo custom', 'z4', 'x5', 'meriva', 'rs6', 'verso', 'touareg', 'shuttle', 'cls class', 'c-max', 'puma', 'cla class', 'i40', 'tiguan allspace', '6 series', 'caravelle', 'karoq', 'i3', 'grand c-max', 't-cross', 'a7', 'golf sv', 'agila', 'gt86', 'yeti', 'california', 'land cruiser', 'edge', 'x6', 'caddy life', '8 series', 'fusion', 'gl class', 'scirocco', 'z3', 'proace verso', 'hilux', 'amarok', 'cc', '7 series', 'avensis', 'eos', 'm class', 'grandland', 'zafira tourer', 'rs5', 'r8', 'mustang', 'antara', 'q8', 'camry', 'clk', 'rs3', 'jetta', 'kadjar', 'sq5', 'rs4', 'supra', 'i8', 'x7', 'sq7', 'g class', 's3', 'crossland', 'tigra', 'escort', 'glb class', 'vivaro', 'verso-s', 'm5', 's4', 'iq', 'a2', 'caddy maxi', 'streetka', 'cascada', 'accent', 's8', 'rs', 'golf s', 'ranger', 'vectra', 'ampera', 'fox', 'urban cruiser', 'm2', 'clc class', 'm6', 's5', 'terracan', "NaN"]

transmission_types = ["semi-auto", "manual", "automatic", "unkown"]
fuel_types = ["petrol", "diesel", "hybrid", "other", "NaN"]

for i in range(len(df)): 
    df.loc[i, "model"] = process.extractOne(df.model[i], models)[0]
    df.loc[i, "transmission"] = process.extractOne(df.transmission[i], transmission_types)[0]
    df.loc[i, "fuelType"] = process.extractOne(df.fuelType[i], fuel_types)[0]

In [63]:
df

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,vw,golf,2016.0,22290,semi-auto,28421.0,petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,toyota,yaris,2019.0,13790,manual,4589.0,petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,audi,q2,2019.0,24990,semi-auto,3624.0,petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,ford,fiesta,2018.0,12500,manual,9102.0,petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,bmw,2 series,2019.0,22995,manual,1000.0,petrol,145.0,42.800000,1.5,97.0,3.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75968,37194,mercedes,c class,2015.0,13498,manual,14480.0,petrol,125.0,53.300000,2.0,78.0,0.000000,0.0
75969,6265,audi,q3,2013.0,12495,semi-auto,52134.0,diesel,200.0,47.900000,2.0,38.0,2.000000,0.0
75970,54886,toyota,aygo,2017.0,8399,automatic,11304.0,petrol,145.0,67.000000,1.0,57.0,3.000000,0.0
75971,860,audi,q3,2015.0,12990,manual,69072.0,diesel,125.0,60.100000,2.0,74.0,2.000000,0.0


In [64]:
# Convert the str NaN values back to pd.NA for easier further processing and readability

df['model'] = df['model'].replace('NaN', pd.NA)
df['transmission'] = df['transmission'].replace('unkown', pd.NA)
df['fuelType'] = df['fuelType'].replace('NaN', pd.NA)

In [65]:
df.isna().sum()

carID                0
Brand             1521
model             1517
year              1491
price                0
transmission       736
mileage           1463
fuelType          1511
tax               7904
mpg               7926
engineSize        1516
paintQuality%     1524
previousOwners    1550
hasDamage         1548
dtype: int64